In [0]:
%sql
Show Catalogs;

#### UDFs, 3 Types :
1. Scalar Python UDF
2. Pandas vectorized UDF
3. UDTF

#### Scalar Python UDF
1. User-defined Functions
2. Take or return Python Objects
3. Operate one row at a time
4. Input or output  is Serialized/Deserialized by pickle or Arrow

#### How to define and use a Python UDF
1. Define a UDF with the following functionality
     - Input -> member_id
     - Output -> Guest/Member

In [0]:
%sql
CREATE TABLE IF NOT EXISTS devscholarsnest.spark_db.members 
AS
SELECT * 
FROM read_files(
  '/Volumes/devscholarsnest/spark_db/datasets/MyLearning/data/members.csv',
  format => 'csv',
  header => 'true',
  inferSchema => 'true'
);

In [0]:
%sql
DESCRIBE EXTENDED devscholarsnest.spark_db.members;

In [0]:
from pyspark.sql.functions import udf

@udf(returnType="string", useArrow=True)
def member_type_udf(id: str):
    if id is None:
        return 'Unknown'
    return 'Guest' if id == 0 else 'Member'


In [0]:
member_df = spark.table("devscholarsnest.spark_db.members")
result_df = member_df.withColumn("member_type",member_type_udf("memid"))
result_df.limit(3).display()

In [0]:
spark.udf.register("member_type_udf",member_type_udf)

In [0]:
%sql
select *, member_type_udf(memid) as member_type
from 
devscholarsnest.spark_db.members
limit 3;

#### Vectorized UDF or Pandas UDF : Comparatively Faster
1. User-defined Functions
2. Take or return Pandas Series/DataFrame
3. Operate Block By Block (Vectorized)
4. Serialized/Deserialized by Arrow

#### How to define and use a Python UDF
1. Define a UDF with the following functionality
     - Input -> member_id
     - Output -> Guest/Member

In [0]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(returnType="string")
def member_type_pudf(id:pd.Series)->pd.Series:
    return id.apply(lambda x: 'Guest' if x==0 else 'Member')

In [0]:
%sql
CREATE TABLE IF NOT EXISTS devscholarsnest.spark_db.bookings 
AS
Select * from read_files(
'/Volumes/devscholarsnest/spark_db/datasets/MyLearning/data/bookings.csv',
format => 'csv',
header => 'true',
inferSchema => 'true'
)

In [0]:
spark.table("devscholarsnest.spark_db.bookings").printSchema()

In [0]:
bookings_df = spark.table("devscholarsnest.spark_db.bookings")

bookings_df.where(member_type_pudf("memid")=="Guest").display()

In [0]:
spark.udf.register("member_type_pudf",member_type_pudf)

In [0]:
%sql
select * from 
devscholarsnest.spark_db.bookings
where member_type_pudf(memid) == "Guest"
limit 3;

In [0]:
from pyspark.sql.functions import expr
result_df = bookings_df.where(expr("member_type_pudf(memid)='Guest'")) 

In [0]:
result_df.limit(3).display()

#### User Defined Table Functions
1. User Defined Function that returns a table
2. Take Python Objects as Input
3. Operates one row at a time
4. Serialized/Deserialized by Arrow or Pickle

#### How to define and use a Python UDF
1. Define a UDF with the following functionality
     - Input -> start_date(ex: 2026-05-18), expand_days(ex:5)
     - Output -> expand the given start date to expand_days as below
     - 2026-05-18
     - 2026-05-19
     - 2026-05-20
     - 2026-05-21
     - 2026-05-22

In [0]:
from pyspark.sql.functions import udtf,col
from datetime import datetime, timedelta

@udtf(returnType="date_value: string",useArrow=True)
class DateExploder:
    def eval(self, start_date:str, expand_days:int):
        current = datetime.strptime(start_date,"%Y-%m-%d")
        for i in range(expand_days):
            yield(current.strftime("%Y-%m-%d"), )
            current += timedelta(days=1)

In [0]:
from pyspark.sql.functions import lit

DateExploder(lit("2026-05-18"), lit(6)).display()

In [0]:
data_schema = "id int, name string, join_date string"

data = [
    (101,"Prashant","2025-05-27"),
    (102,"Sakshi","2025-05-18"),
    (103,"Mayank","2025-05-19")
]

df = spark.createDataFrame(data,data_schema)

UDTFs can be used only in Jon=in statements, cannot be used in select statement.

In [0]:
result_df = (df.lateralJoin(DateExploder(col("join_date"),lit(7)))
                .selectExpr("id","name","join_date","date_value as attendance_date")
                )

result_df.display()

In [0]:
spark.udtf.register("date_exploder",DateExploder)

In [0]:
df.createOrReplaceTempView("students_view")

In [0]:
%sql
select * 
from students_view, lateral date_exploder(join_date,6)

#### EaseWithData

**Working of Python UDF on Cluster**
- Spark application assume is running on Cluster with 1 Driver and 2 Nodes.
- Once we write this UDF to process our logic, Spark copies this UDF to individual worker nodes.
- Once this is available spark spins up the Python Process and once this process is ready data is serialized in the format that Python can understand, Data is Processed Row by Row
- Once data is processed results are reverted back to JVM Process.
- Then JVM reports back to driver with the results.
- Python Process resides in the same node where JVM resides, if there are many UDFs written in an application and python process grows in size and exceeds memory limits we will run out of memory on the Node.
- Demerits :
     - Data needs to be Serialized/Deserialized
     - Python Process has to be spin up.
     _ Processing data row by row by python process.
- Solution :
     - Write UDF in Java or Scala once written register this UDF, 
     - Since we register the UDF already present in JVM , We dont need to spin up Python Process.
     - We can now reuse the Java/Scala UDF within python code itself.

In [0]:
df = (spark.read
      .format("csv")
      .option("header","true")
      .option("inferSchema","true")
      .load("/Volumes/easewithdata/spark_db/data/emp.csv"))

First Way

In [0]:
def bonus(salary):
    return int(salary)*0.1;

In [0]:
from pyspark.sql.functions import udf

bonus_udf = udf(bonus)

In [0]:
df.withColumn("Bonus",bonus_udf(df.salary)).display()

Second Way 

In [0]:
from pyspark.sql.functions import udf

@udf(returnType="double", useArrow=True)
def bonus(salary:str):
    return int(salary)*0.1

In [0]:
spark.udf.register("bonus_sql",bonus)

In [0]:
df.withColumn("Bonus",bonus(df.salary)).display()